In [ ]:
BANNER = \
"""# ----------------------------------------------------------------------
# PTS | Paulo Trigo Silva
# mLDm | MoP
# v01
# ----------------------------------------------------------------------
"""

In [ ]:
# ------------------------------------------------------------
# from modelling datasets to learned models
# ------------------------------------------------------------

# MoP_04 | From Dataset to Model
# notebook 2 | learning and prediction

# input:
#   03_output/penguins_modelling_[A-E]_*.csv
#   NOTE: the input of this notebook is the output of the previous one
#
# outputs:

# goal:
#   learn, predict and evaluate models using five modelling
#   datasets, comparing learning methods and prediction contexts

In [ ]:
# ------------------------------------------------------------
# define input and output paths
# ------------------------------------------------------------

from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "02_code": ROOT = ROOT.parent

INPUT = ROOT / "01_input"
OUTPUT = ROOT / "03_output"

assert INPUT.exists(), f"PTS | missing input directory: {INPUT}"
OUTPUT.mkdir( parents=True, exist_ok=True )

print( "input:", INPUT )
print( "output:", OUTPUT )

In [ ]:
# ------------------------------------------------------------
# load modelling datasets
# ------------------------------------------------------------

import pandas as pd

# input filenames for each prediction scenario
# TODO | define each scenario name and corresponding dataset file
filenames = {
  "A": ...
  ...
  "E": ...,
  "A_noOHE": ...
}

# load the five modelling datasets
datasets = {}

for scenario, filename in filenames.items():
  path = OUTPUT / filename
  assert path.exists(), f"PTS | missing input file: {filename}"

  # TODO | load the modelling dataset from its CSV file
  datasets[ scenario ] = ...
  print( f"{scenario}: {datasets[ scenario ].shape} — {filename}" )

In [ ]:
# ------------------------------------------------------------
# validate modelling datasets
# ------------------------------------------------------------

# validation uses scenario A as reference and compares it with all other scenarios, checks:
# - number of observations
# - technical ID uniqueness
# - observation correspondence
# - target consistency, and
# - absence of missing values
#
# NOTE:
# technical IDs are assigned sequentially in each modelling dataset
# since all scenarios preserve the original row order,
# the same ID identifies the same original observation across scenarios

# use scenario A as reference
reference = datasets[ "A" ]
for scenario, modelling_dataset in datasets.items():
  # verify observation identifiers
  assert len( modelling_dataset ) == 333, \
    f"PTS | unexpected number of observations in scenario {scenario}"

  assert modelling_dataset[ "ID" ].is_unique, \
    f"PTS | duplicated ID in scenario {scenario}"

  assert modelling_dataset[ "ID" ].equals( reference[ "ID" ] ), \
    f"PTS | inconsistent observation IDs in scenario {scenario}"

  # verify target consistency
  assert modelling_dataset[ "species" ].equals( reference[ "species" ] ), \
    f"PTS | inconsistent target values in scenario {scenario}"

  # verify missing values
  assert not modelling_dataset.isna().any().any(), \
    f"PTS | missing values in scenario {scenario}"

print( "modelling datasets of all scenarios: valid" )

### [A] Orange DM - Comparing Decision Trees with and without One-Hot Encoding

**Research question:** How does feature representation affect the structure and predictive performance of a decision tree?

Create a new Orange DM workflow and save it as:
- `02_code/penguins_OHE_comparison.ows`.

We start with **Tree** because it supports categorical attributes directly and produces an interpretable model. Other learners may apply internal transformations, making the effect of OHE more difficult to isolate.

#### [A] 1. Prepare the two representations

Add two **File** widgets and load:

- **Without OHE:** `03_output/penguins_modelling_A_study_noOHE.tab`
- **With OHE:** `03_output/penguins_modelling_A_study.tab`

Both datasets contain the same observations and target (`species`). They differ only in the representation of `island` and `sex`.

The `.tab` files already define attribute types and roles: `species` is the target and `ID` is a meta attribute.

#### [A] 2. Learn and inspect the decision trees

- Connect each **File** to a separate **Tree** widget, using identical learner settings.
- Connect each **Tree** model output to a **Tree Viewer**.
- Compare the root attribute, partitions, tree size and depth.

**NOTE:** Train both trees using all observations. Structural differences are possible, but not guaranteed.

#### [A] 3. Evaluate predictive performance

For each representation:

- Connect **File** and the corresponding **Tree** learner to **Test & Score**.
- Select *Random Sampling*: 80% training, 20% testing, with stratification.
- Connect **Test & Score** to **Confusion Matrix**.
- Inspect classification accuracy and errors.


**IMPORTANT — Experimental limitation:**

The two Test & Score widgets perform independent train/test sampling. Although both use the same sampling proportions and stratification, we do not explicitly control which observations are assigned to each partition.

Using the same random seed would not be sufficient without also ensuring identical observation ordering and sampling procedures.

Consequently, differences in accuracy may result from:
- Feature representation (with or without OHE).
- Different observations selected for training and testing.

**A controlled comparison requires identical train/test partitions**, which we will explicitly establish using observation IDs in Python (to be explored later in this notebook).


#### [A] 4. Interpret the results

Discuss:

- Did OHE change the learned tree structure?
- Did the two evaluations produce different accuracies?
- Can we conclude that one representation performs better?
- What experimental control is missing for a fair comparison?

**Conclusion:** This experiment illustrates both the influence of feature representation on model learning and the importance of controlled evaluation. A rigorous performance comparison requires identical train/test partitions, which we will implement later in Python.



**NOTE — model training vs. model evaluation:**

- **Tree Viewer** displays the model trained using all available observations.
- **Test & Score** trains new models using only the training partition and evaluates them on the test partition.
- ... therefore, the trees displayed in Tree Viewer are not the models evaluated by Test & Score.

### [B] Orange DM — Comparing Scenarios, Learners and Predictive Performance

**Research question:** How do prediction scenarios and learning algorithms affect predictive performance?

Create a new Orange DM workflow and save it as:
- `02_code/penguins_learners_comparison.ows`.

#### [B] 1. Load the five prediction scenarios

Add five **File** widgets and load:

- `penguins_modelling_A_study.tab`
- `penguins_modelling_B_anywhere.tab`
- `penguins_modelling_C_whenever.tab`
- `penguins_modelling_D_biological.tab`
- `penguins_modelling_E_morphological.tab`

All datasets contain the same observations and target (`species`), but differ in their available features.

#### [B] 2. Compare three learners

From the **Model** category, add:

- **Tree** — decision tree classifier
- **kNN** — nearest-neighbor classifier
- **Naive Bayes** — probabilistic classifier

For each scenario:

- Connect **File** to a separate **Test & Score** widget.
- Connect the three **Learner** outputs to that Test & Score.
- Select *Random Sampling*: 80% training, 20% testing, with stratification.
- Set *Repeat train/test* to 1.
- Connect **Test & Score** to **Confusion Matrix**.

Use identical learner and evaluation settings across all five scenarios.

#### [B] 3. Analyze the results

Compare:

- Which learner achieves the highest classification accuracy in each scenario?
- How does accuracy change when contextual attributes are excluded?
- Which classes account for most classification errors?
- Does the preferred learner depend on the prediction context?

**NOTE — Learner preprocessing:**

Orange learners may apply different internal preprocessing. In particular, Naive Bayes discretizes continuous attributes, while kNN uses numerical distances and preprocessing. The comparison therefore concerns complete learning procedures, not only their underlying algorithms.

#### [B] 4. Understand experimental control

**Within each Test & Score:**

The three learners are trained and evaluated using the same train/test partitions. Their predictive performance can therefore be compared under equivalent data-sampling conditions.

**Across different Test & Score widgets:**

Each scenario is evaluated independently. Even with identical sampling proportions, stratification and repetition settings, the widgets do not explicitly guarantee identical train/test observations.

Consequently, differences in accuracy across scenarios may reflect both feature selection and differences in the observations used for training and testing.

**Conclusion:** Comparing learners within the same evaluation is more controlled than comparing independently evaluated datasets. In Python, we will use observation IDs to establish identical train/test partitions across all scenarios.

In [ ]:
# ------------------------------------------------------------
# define train/test holdout
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split

# after the Orange DM experiments, we introduce explicit experimental control:
# - all scenarios and feature representations must
# - use exactly the same training and test observations

# split observation IDs rather than features (X) and target (y)
# - this allows the same train/test observations to be used across all modelling datasets
# - ensuring a controlled comparison

# also:
# - stratification (stratify) preserves approximately the original class proportions
# - RANDOM_SEED ensures reproducibility of the partition
RANDOM_SEED = 42
reference = datasets[ "A" ]

# TODO | create a reproducible, stratified 80/20 train/test split
# using the ID column to preserve the same observations across scenarios
train_ID, test_ID = train_test_split(
  reference[ "ID" ],
  ...
  stratify=...
  random_state=... )

print( "train:", len( train_ID ) )
print( "test:", len( test_ID ) )

# inspect class distributions
# isin( train_ID ) returns a boolean Series:
# - True for each row whose ID is in train_ID, and False otherwise
# this boolean mask selects only the corresponding training observations
train = reference[ reference[ "ID" ].isin( train_ID ) ]
# TODO | select test observations using their IDs
test = ...

print( "\ntrain class distribution:" )
print( train[ "species" ].value_counts() )

print( "\ntest class distribution:" )
print( test[ "species" ].value_counts() )


# expected output (partial):
# train: 266
# test: 67
# test class distribution: Adelie 29, Gentoo 24, Chinstrap 14

In [ ]:
# ------------------------------------------------------------
# prepare features and target for learning
# ------------------------------------------------------------

# prepare the same train/test observations for all datasets
# ID is used to select observations, but not as a model feature
# species is the target variable

partitions = {}

for scenario, data in datasets.items():
  train = data[ data[ "ID" ].isin( train_ID ) ]
  test = data[ data[ "ID" ].isin( test_ID ) ]

  X_train = train.drop( columns=[ "ID", "species" ] )
  # TODO | prepare test features using the same columns as for training
  X_test = ...

  # TODO | extract the target values for train and test observations
  ...

  partitions[ scenario ] = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test
  }

  print( f"{scenario}: X_train={X_train.shape}, X_test={X_test.shape}" )

# expected output (partial):
# A: X_train=(266, 10), X_test=(67, 10)
# B: X_train=(266, 7), X_test=(67, 7)
# A_noOHE: X_train=(266, 7), X_test=(67, 7)

In [ ]:
# ------------------------------------------------------------
# define classifier training and evaluation
# ------------------------------------------------------------

from sklearn.metrics import accuracy_score

# train, predict and evaluate using numerical features
# no categorical encoding is performed
# datasets containing categorical values (e.g., A_noOHE)
# require encoding before using the selected classifiers

def train_evaluate( scenario, partitions, classifier_type, classifier_parameters,
                    scaler=None ):
  # scenario: identifier of the modelling dataset
  # partitions: training and test data for each scenario
  # classifier_type: scikit-learn classifier class
  # classifier_parameters: dictionary of classifier initialization parameters
  # scaler: optional feature scaling method (None means no scaling)

  data = partitions[ scenario ]

  X_train = data[ "X_train" ]
  X_test = data[ "X_test" ]
  y_train = data[ "y_train" ]
  y_test = data[ "y_test" ]

  # scale features when requested
  # - learn scaling parameters from training data only
  if scaler is not None:
    scaler.fit( X_train )
    # apply the learned transformation to both partitions
    X_train = scaler.transform( X_train )
    X_test = scaler.transform( X_test )


  # create and train the classifier
  classifier = classifier_type( **classifier_parameters )
  classifier.fit( X_train, y_train )

  # predict and evaluate
  # TODO | identify the test features to use for prediction
  y_pred = classifier.predict( ...
  # TODO | evaluate predictive accuracy using the test targets
  accuracy = accuracy_score( y_test, ...

  return classifier, y_pred, accuracy

In [ ]:
# ------------------------------------------------------------
# first experiment: decision tree on scenario A
# ------------------------------------------------------------

from sklearn.tree import DecisionTreeClassifier

scenario = "A"

classifier, y_pred, accuracy = train_evaluate(
  scenario,
  partitions,
  DecisionTreeClassifier,
  { "random_state": RANDOM_SEED } )

print( "scenario:", scenario )
print( "classifier:", type( classifier ).__name__ )
print( f"accuracy: {accuracy:.2%}" )

In [ ]:
# ------------------------------------------------------------
# compare classifiers on scenario A
# ------------------------------------------------------------

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

scenario = "A"

# define classifiers and their initialization parameters
#
# the Tree "random_state" ensures reproducible split selection
# - when multiple equally good partitions are available
#
# NOTE: RANDOM_SEED controls two distinct random processes:
# - train_test_split: selection of training and test observations
# - DecisionTreeClassifier: internal randomness in tree construction
# for simplicity, the same seed is used for reproducibility,
# - but the processes are independent
classifiers = {
  "Tree": ( DecisionTreeClassifier, { "random_state": RANDOM_SEED } ),
  # TODO | configure a kNN classifier using five neighbors
  "kNN": ...,
  "Naive Bayes": ( GaussianNB, {} )
}


# all classifiers use the same training and test observations
# only the learning algorithm and its parameters vary

# NOTE: numerical features are not scaled at this stage
# this may particularly affect distance-based classifiers such as kNN

# train and evaluate each classifier
for name, ( classifier_type, classifier_parameters ) in classifiers.items():
  classifier, y_pred, accuracy = train_evaluate( scenario, partitions,
                                                 classifier_type, classifier_parameters )

  print( f"{name}: {accuracy:.2%}" )


# unlike the earlier Orange DM experiments, this comparison
# explicitly controls the train/test partition through observation IDs
#
# NOTE: Orange Naive Bayes and scikit-learn GaussianNB use different
# treatments of numerical features, so their results need not match

# expected output:
# Tree: 94.03%
# kNN: 73.13%
# Naive Bayes: 83.58%

In [ ]:
# ------------------------------------------------------------
# compare classifiers across prediction scenarios
# ------------------------------------------------------------

# evaluate the same classifiers across five prediction scenarios
# the train/test observations and classifier settings remain fixed
# only the available features vary between scenarios

# the A_noOHE is excluded because it contains categorical values
# that require additional encoding before learning

results = []

for scenario in [ "A", "B", "C", "D", "E" ]:
  # TODO | iterate over all configured classifiers
  for ... in classifiers.items():
    # TODO | call train_evaluate with the current scenario and classifier configuration
    classifier, y_pred, accuracy = train_evaluate( ...
    results.append( {
      "scenario": scenario,
      "classifier": name,
      "accuracy": accuracy } )

# organize results for comparison
results = pd.DataFrame( results )

comparison = results.pivot(
  index="scenario",
  columns="classifier",
  values="accuracy" )

display( comparison.round( 3 ) )

# NOTE: differences across scenarios can now be attributed to
# the available feature representations under the fixed evaluation setup
# this does not imply that more features necessarily improve accuracy

# expected output (partial data):
# classifier  Naive Bayes   Tree    kNN
# scenario
# A                 0.836  0.940  0.731
# B                 0.955  0.910  0.731
# ...
# E                 0.955  0.910  0.731

In [ ]:
# ------------------------------------------------------------
# evaluate the effect of feature scaling on kNN
# ------------------------------------------------------------

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler

scenario = "A"

# compare the original features with two scaling methods
# the classifier, its parameters and train/test partitions remain fixed

# TODO | configure the min-max and standard scaling methods
scalers = {
  "original": None,
  "min-max": ...,
  "standard": ...
}

results_scaling = []

for name, scaler in scalers.items():
  # TODO | evaluate kNN with 5 neighbors using the current scaling method
  classifier, y_pred, accuracy = train_evaluate( ...
      
  results_scaling.append( {
    "scaling": name,
    "accuracy": accuracy
  } )

display( pd.DataFrame( results_scaling ).round( 3 ) )

# NOTE: scaling is applied to all features, including OHE indicators
# different scaling methods may change their contribution to distances

# expected output:
#     scaling  accuracy
# 0  original     0.731
# 1   min-max     1.000
# 2  standard     1.000

In [ ]:
# ------------------------------------------------------------
# compare kNN scaling across prediction scenarios
# ------------------------------------------------------------

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# compare the effect of scaling across five prediction scenarios
# the classifier, its parameters and train/test observations remain fixed
# only the prediction scenario and scaling method vary

# TODO | configure the scaling methods
scalers = {
  ...
}

results_scaling = []

for scenario in [ "A", "B", "C", "D", "E" ]:
  # TODO | iterate over the configured scaling methods
  for ... in scalers.items():
    # TODO | evaluate kNN with 5 neighbors using the current scenario and scaling method

    results_scaling.append( {
      "scenario": scenario,
      "scaling": name,
      "accuracy": accuracy
    } )

# compare accuracy across scenarios and scaling methods
results_scaling = pd.DataFrame( results_scaling )

comparison = results_scaling.pivot(
  index="scenario",
  columns="scaling",
  values="accuracy" )

display( comparison.round( 3 ) )

# NOTE: scaling is applied to all features, including OHE indicators
# scaling may change the relative contribution of features to distances
# higher accuracy in one holdout does not establish better generalization

# expected output (partial data and format):
# scenario A: original=0.731, min-max=1.000, standard=1.000
# scenario B: original=0.731, min-max=0.970, standard=0.985

In [ ]:
# ------------------------------------------------------------
# compare scenarios, classifiers and scaling methods
# ------------------------------------------------------------

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# compare five scenarios, three classifiers and three scaling methods
# all configurations use the same training and test observations
# classifier hyperparameters remain fixed across configurations

scalers = {
  "original": None,
  "min-max": MinMaxScaler(),
  "standard": StandardScaler()
}

results_all = []

for scenario in [ "A", "B", "C", "D", "E" ]:
  for name, ( classifier_type, classifier_parameters ) in classifiers.items():

    # TODO | iterate over all configured scaling methods
    for ... in scalers.items():

      # TODO | evaluate the current scenario, classifier and scaling method
      classifier, y_pred, accuracy = train_evaluate( ... )

      # TODO | record the current experimental configuration and its accuracy
      results_all.append( {
        "scenario": ...
        "classifier": ...
        "scaling": ...
        "accuracy": ...
      } )

# organize results for comparison
results_all = pd.DataFrame( results_all )

comparison = results_all.pivot(
  index=[ "scenario", "classifier" ],
  columns="scaling",
  values="accuracy" )

display( comparison.round( 3 ) )

# NOTE: scaling can affect classifiers differently
# - kNN is a distance-based method, so particularly sensitive to scaling
# - Tree is largely insensitive to monotonic feature scaling
# - GaussianNB is theoretically invariant to linear rescaling and shifting
#   but scikit-learn uses variance smoothing (var_smoothing)
#   which can change predictions significantly after scaling

# NOTE: scaling is applied to all features, including OHE indicators
# all 45 configurations (5 scenarios x 3 classifiers x 3 scaling methods)
# use the same controlled train/test holdout

# NOTE: a single holdout provides one estimate of predictive performance
# - results may depend on the particular observations selected for testing
# - comparing many configurations on the same test set may also lead
#   to overestimating the performance of the selected configuration
#
# alternative evaluation procedures include:
# - repeated holdout: evaluates models across multiple random partitions
# - k-fold cross-validation: evaluates models across K complementary folds
# - repeated k-fold: repeats cross-validation with different fold assignments

# expected output (partial):
# scaling               min-max  original  standard
# scenario classifier
# A        Naive Bayes    0.687     0.836     0.687
#          Tree           0.940     0.940     0.940
#          kNN            1.000     0.731     1.000
# B        Naive Bayes    0.955     0.955     0.955
# ...

In [ ]:
# ------------------------------------------------------------
# reflect on modelling and evaluation — questions to answer
# ------------------------------------------------------------

# answer the following questions in the next cell
# use the experiments and results obtained in this notebook

# question 1: why can Orange Tree use A_noOHE directly,
# while the selected scikit-learn classifiers cannot?

# question 2: how does predictive accuracy vary across classifiers
# when the scenario and train/test partitions remain fixed?

# question 3: why is kNN more sensitive to feature scaling
# than Tree and GaussianNB?

# question 4: why must scaling parameters be learned
# exclusively from training observations?

# question 5: why is using identical train/test partitions important
# when comparing different model configurations?

# question 6: how does this notebook ensure that exactly the same
# observations are used for training and testing across all scenarios?

# question 7: why does higher accuracy on a single test partition
# not guarantee better generalization?

# question 8: how can repeated holdout or cross-validation
# improve the reliability of model evaluation?

### (some) Theoretical Considerations
### Categorical Features — Orange DM vs. scikit-learn

**Why is `A_noOHE` excluded from the scikit-learn experiments?**

The `A_noOHE` dataset preserves the original nominal attributes:

- `island`: Biscoe, Dream, Torgersen
- `sex`: FEMALE, MALE

These attributes contain categorical values rather than numerical measurements.

#### 1. Categorical support depends on the learner implementation

| Learner | Orange DM | scikit-learn |
|:---|:---|:---|
| Decision Tree | Supports categorical attributes directly | `DecisionTreeClassifier` requires numerical features |
| kNN | Automatically encodes categorical attributes | `KNeighborsClassifier` requires numerical features |
| Naive Bayes | Supports discrete attributes | `GaussianNB` requires numerical features |

**NOTE:** This is an implementation distinction, not a general limitation of the learning algorithms. For example, decision trees can partition categorical attributes directly, as demonstrated in Orange DM.

#### 2. Why not simply convert categories into integers?

We could represent `island` values using integer codes:

| value | code |
|:---|:---|
| Biscoe | 0 |
| Dream | 1 |
| Torgersen | 2 |

However, these codes introduce an artificial numerical ordering.

For distance-based methods such as kNN, this also introduces artificial distances between categories. For decision trees, numerical thresholds would impose ordered partitions rather than directly partitioning nominal values.

**One-Hot Encoding (OHE)** avoids introducing such an ordering by representing each category with a binary indicator.

#### 3. Our experimental choice

- **Orange DM:** compare scenario A with and without OHE using Tree, which supports categorical attributes directly.
- **scikit-learn:** compare the five scenarios using the previously generated OHE representations.
- **A_noOHE:** exclude from the scikit-learn experiments because additional categorical encoding would be required.

This choice keeps the experiments focused on the original learning objectives without introducing further feature transformations.

**Conclusion:** Feature representation must be compatible with the selected learner implementation. The same learning algorithm may handle categorical attributes differently across software libraries.
